# S&P 500 (daily): data and MGD model
Data: the 6062 daily log-returns $X(u) = \log P(u) - \log P(u-1)$ of the S&P 500, standardized once; the whole series is **one** sample ($d = 6062$).
Model: run `snp_M0` (wavelet scattering moments of order 1 to 4, $J = 6$, $Q = 1$, Morlet, periodic boundaries), 5 seeds (900–904), 100 generated series per seed, $\sigma = 3.5$.
$\Theta$ = $\Theta_{\rm reg}$ at the last time step ($t \approx 1$), $\lambda$ = cross-seed choice (`snp_xseed.sh`, window `end0.99`). M1–M3 only appear in the energy table.

Run on Jean Zay from `snp_experiment/`. Needs `figures/logp_snp/logp_snp_M*.npz` (`python snp_experiment/logp_snp.py`). Figures: `figures/talk/`.

Error bands. **Data**: standard error over the 23 blocks of 256 days (value computed on the whole series). **Model**: mean over the 100 series of each seed, then mean ± std over the 5 seeds.

In [ ]:
%matplotlib inline
import sys
from pathlib import Path
ROOT = Path.cwd()                                   # snp_experiment/
for p in ('codes', 'codes/filters', 'codes/potentials', 'data', 'turbulence', 'jets', 'snp_experiment'):
    sys.path.insert(0, str(ROOT.parent / p))
import json
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from IPython.display import display
import talk_figures as tf
import run_comparison as rc
import jets_compare as jc
import snp_stats as ss
from snp_loader import load_snp_returns
from logp_family_contrib import load_potentials, phi_matrix
tf.set_output(ROOT / 'figures' / 'talk')          # every figure is also saved here as PNG

In [ ]:
LABEL, SEEDS, WINDOW = 'snp_M0', [900, 901, 902, 903, 904], 'end0.99'
configs = jc.find_seed_configs(ROOT, LABEL, SEEDS)
assert all(configs.values()), configs
runs = {s: rc.load_config(c, ROOT, label=LABEL, extras=False) for s, c in configs.items()}
cfg = runs[SEEDS[0]]['cfg']
x1, raw, dates = load_snp_returns()
x1 = x1.reshape(1, -1).numpy()                                        # data, (1, 6062), standardized
xts = [r['xt'].numpy() for r in runs.values()]                        # generated, one (100, 6062) array per seed
xt = xts[0]
xsel = torch.load(ROOT / 'results' / 'xseed' / f'xseed_{LABEL}.pt', map_location='cpu', weights_only=False)
LAM = xsel['selected'][WINDOW]
sol = torch.load(xsel['theta_files'][WINDOW][configs[SEEDS[0]]], map_location='cpu', weights_only=False)
assert sol['config'] == configs[SEEDS[0]] and float(sol['lam']) == float(LAM)
theta = sol['Theta_reg'][-1].double().numpy()
print('data', x1.shape, f'({dates[0].date()} to {dates[-1].date()}) | generated', xt.shape, '| seeds', SEEDS, '| lambda', LAM)

## Data and generated series
Daily log-returns (standardized): the data (top) and three generated series. Volatility clustering and the large negative days are the features to compare.

In [ ]:
rng = np.random.default_rng(0)
idx = rng.choice(len(xt), 3, replace=False)
fig, axes = plt.subplots(4, 1, figsize=(13, 7.5), sharex=True, sharey=True)
axes[0].plot(x1[0], color=tf.DATA, lw=0.7); axes[0].set_ylabel('S&P data')
for ax, i in zip(axes[1:], idx):
    ax.plot(xt[i], color=tf.MODEL, lw=0.7); ax.set_ylabel(f'generated {i}')
axes[-1].set_xlabel('u (trading days)')
fig.tight_layout(); tf._save(fig, 'series_data_vs_generated')

## Wavelet coefficients of the data, per scale
Histogram of the real part of the Morlet coefficients (Q = 1, J = 6) of the data; dashed: Gaussian.

In [ ]:
tf.plot_wavelet_histograms(x1, int(cfg['J']), 'data_wavelet_histograms')

## Wavelet coefficients: data vs generated
Same histograms for the 100 series of seed 900 (blue) against the data (black), pooled over the scales' coefficients of each series.

In [ ]:
from filters_bank import return_Filters
J, M = int(cfg['J']), x1.shape[-1]
Fl = return_Filters(M, J, 1, device='cpu').reshape(-1, M).to(torch.complex128)[:-1]
def coefs(x):
    z = torch.fft.ifft(torch.fft.fft(torch.as_tensor(x))[:, None, :] * Fl[None], dim=-1).real.numpy()
    return z                                                                     # (n, J, M)
zd, zg = coefs(x1.astype(np.float64)), coefs(xt.astype(np.float64))
fig, axes = plt.subplots(2, 3, figsize=(15, 8), sharey=True)
for j, ax in enumerate(axes.flat):
    sd = zd[:, j].std()
    bins = np.linspace(-8, 8, 121)
    hd, _ = np.histogram(zd[:, j].ravel() / sd, bins, density=True)
    hg, _ = np.histogram(zg[:, j].ravel() / sd, bins, density=True)
    c = 0.5 * (bins[1:] + bins[:-1])
    ax.plot(c[hd > 0], hd[hd > 0], color=tf.DATA, label='S&P data'); ax.plot(c[hg > 0], hg[hg > 0], color=tf.MODEL, label='MGD model')
    ax.set_yscale('log'); ax.set_ylim(1e-5, 1); ax.set_title(f'j = {j}')
axes[0, 0].legend(); fig.tight_layout(); tf._save(fig, 'wavelet_histograms_data_vs_generated')

## Statistics of the S&P returns: data and model
Computed once (`snp_stats.compute`): CDF of $X$, $\mathrm{Ave}_u|\delta_s p(u)|^q$ with $\delta_s p(u) = X(u-s+1)+\dots+X(u)$ at $s = 2^j$ days, signed leverage, Zumbach moment, 5-day rolling volatility, power spectra (returns and volatility).

In [ ]:
S = ss.compute(x1, xts)

### Distribution of the returns
Both tails of the CDF of $X$ in log scale.

In [ ]:
ss.plot_cdf(S['cdf'], 'sp_cdf')

### Increments over $s = 2^j$ days: $\mathrm{Ave}_u |\delta_s p(u)|^q$
Orders $q = 1, 2, 4$. Blocks of 256 days reach $s \le 64$ only: the data band stops there.

In [ ]:
ss.plot_increments({1.0: S['inc1'], 2.0: S['inc2'], 4.0: S['inc4']}, 'sp_increments')

### Leverage and Zumbach
Signed leverage $L(\tau) = \langle X(t-\tau) X(t)^2 \rangle / \langle X^2 \rangle^2$: negative for $\tau > 0$ in the data (a loss today raises tomorrow's volatility). Zumbach: past 5-day squared move against present volatility; time-reversible series are symmetric in $\tau$.

In [ ]:
ss.plot_leverage_zumbach(S['leverage'], S['zumbach'], 'sp_leverage_zumbach', scale=5)

### 5-day rolling volatility
Histogram of $\log_{10}\sqrt{\frac15\sum_{5\,\rm days} X^2}$.

In [ ]:
ss.plot_rolling_vol(S['vol5'], 'sp_rolling_vol')

### Power spectra
Returns: flat for white noise. Volatility ($|X| - \langle|X|\rangle$): long memory shows as a power law at low frequency. Data bands stop at $f = 1/256$.

In [ ]:
ss.plot_spectra(S['spec_ret'], S['spec_vol'], 'sp_spectra')

## Distribution of −log p of the generated series
$-\log p(x) = -\Theta\cdot\phi(x)$ + constant, evaluated on the 5 × 100 generated series (histogram) and on the data (vertical line); the constant is arbitrary, so values are relative to the median of the generated ones.

In [ ]:
D = np.load(ROOT / 'figures' / 'logp_snp' / 'logp_snp_M0.npz')
assert [int(s) for s in D['seeds']] == SEEDS and abs(float(D['lam']) - LAM) < 1e-12
Cg, Cd = D['C_gen'], D['C_data']                      # (seeds, families + total, n_gen), (seeds, families + total)
nlp = -Cg[:, -1]                                      # -log p of generated, per seed
ref = np.median(nlp, axis=1, keepdims=True)
gen = (nlp - ref).ravel(); dat = -Cd[:, -1] - ref[:, 0]          # data, one value per seed
from scipy.stats import gaussian_kde
grid = np.linspace(gen.min() - 50, gen.max() + 50, 400)
fig, ax = plt.subplots(figsize=(8, 4.6))
ax.hist(gen, bins=30, color=tf.MODEL, alpha=0.25, density=True)
ax.plot(grid, gaussian_kde(gen)(grid), color=tf.MODEL, label='generated (5 seeds x 100), kernel density')
m, sd = dat.mean(), dat.std(ddof=1)
ax.axvspan(m - sd, m + sd, color=tf.DATA, alpha=0.2, lw=0)
ax.axvline(m, color=tf.DATA, lw=2.2, label=f'S&P data: mean ± std over seeds')
ax.set_xlabel(r'$-\log p$ (relative to the median of generated)'); ax.set_ylabel('density'); ax.legend()
fig.tight_layout(); tf._save(fig, 'neglogp_generated_vs_data')
print('data -log p, relative to median of generated, per seed:', np.round(dat, 2), '| generated std', gen.std().round(3))

## Contribution of each potential family to −log p (generated series)
Top: mean; middle: std across series; bottom: std divided by the std of the total. Error bars: std over the 5 seeds. The data value (one signal) is in the previous figure.

In [ ]:
import re
fams = [str(f) for f in D['families']]
rank = lambda f: next(i for i, (pat, _) in enumerate(tf._LABELS) if re.search(pat, f))     # scalar, scat1, scat2, scat3 Re/Im, scat4 Re/Im
order = sorted(range(len(fams)), key=lambda i: rank(fams[i]))                              # the builder creates scat2 after scat3
tf.plot_energy_bars({**{fams[i]: Cg[:, i] for i in order}, 'TOTAL': Cg[:, -1]}, 'logp_per_family', Q=cfg['Q'])

## Number of statistics in the model

In [ ]:
exp = rc._find_exp_dir(ROOT, configs[SEEDS[0]])
pots = load_potentials(exp, cfg, 'cpu')
n_stats = tf.count_statistics(pots, torch.as_tensor(x1[:, None, :]).float(), theta, Q=cfg['Q'])
n_stats.to_csv(ROOT / 'figures' / 'talk' / 'number_of_statistics.csv')
display(n_stats)

## Energy table: categories of statistics, models M0–M3
Same table as the turbulence and jets notebooks, over the generated series; one block per model.

In [ ]:
tabs = {}
for m in ('M0', 'M1', 'M2', 'M3'):
    f = ROOT / 'figures' / 'logp_snp' / f'logp_snp_{m}.npz'
    if not f.exists():
        print(m, 'MISSING:', f); continue
    Dm = np.load(f)
    fm = [str(x) for x in Dm['families']]
    tabs[m] = tf.energy_table({k: Dm['C_gen'][:, i] for i, k in enumerate(fm)}, {k: int(w) for k, w in zip(fm, Dm['widths'])}, f'energy_table_{m}')
    display(tabs[m])
tf.plot_energy_table(tabs, 'energy_table')